# 5-2절 연습 문제 풀이

이 노트북은 5-2절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch05/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
DATA_ROOT = '../../downloads'

def loaders(dataset='MNIST', batch_size=64, transform=None, train_transform=None):
    cls = getattr(datasets, dataset)
    transform = transform or transforms.ToTensor()
    full = cls(root=DATA_ROOT, train=True, download=True,
               transform=train_transform or transform)
    test_set = cls(root=DATA_ROOT, train=False, download=True, transform=transform)
    n_valid = int(len(full) * 0.2)
    g = torch.Generator().manual_seed(SEED)
    tr, va = random_split(full, [len(full) - n_valid, n_valid], generator=g)
    return (DataLoader(tr, batch_size=batch_size, shuffle=True),
            DataLoader(va, batch_size=batch_size),
            DataLoader(test_set, batch_size=batch_size))

def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    tot = correct = n = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = criterion(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            tot += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item(); n += y.size(0)
    return tot / n, correct / n * 100

def fit(model, epochs=5, lr=1e-3, dataset='MNIST', **kw):
    tr, va, te = loaders(dataset, **kw)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(1, epochs + 1):
        trl, tra = run_epoch(model, tr, criterion, optimizer)
        val, vaa = run_epoch(model, va, criterion)
        print(f'  {e}/{epochs} 훈련 {trl:.4f} / 검증 {val:.4f} ({vaa:.2f}%)')
    tel, tea = run_epoch(model, te, criterion)
    print(f'  평가 정확도 {tea:.2f}%')
    return tea

class ConvBlock(nn.Module):
    def __init__(self, fan_in, fan_out):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(fan_in, fan_out, kernel_size=3, stride=1, padding=1),
            nn.ReLU(), nn.MaxPool2d(2))
    def forward(self, x): return self.block(x)

## 연습 5-4

[그림 5-1]과 [그림 5-11]의 혼동 행렬을 비교해 보면 7을 2로 잘못 분류한 패턴이 12건에서 7건으로 줄었다. 다음은 합성곱 신경망 모델에서도 여전히 7을 2로 잘못 분류하는 7건의 숫자 이미지이다.

그림 5-13 합성곱 구조의 숫자 분류기 모델이 2로 잘못 분류한 숫자 7의 샘플

7개의 샘플에서는 숫자 7 중간에 짧은 가로선을 긋는 공통된 패턴이 보인다. 실제로 이렇게 쓰는 사람이 일정 비율 있다.

다층 퍼셉트론에서 7을 9로 잘못 분류했지만 합성곱 신경망에서 7로 분류한 이미지의 상당수가 이런 패턴을 보이고 있다. 그런데 같은 가로선 패턴인데도 7을 2로 잘못 분류한 경우는 줄지 않고 여전히 남아 있는 이유는 무엇일까?

이런 패턴을 제대로 분류하는 숫자 분류기를 만들고자 한다면 어떻게 접근해야 할까?

### 풀이

**7을 2로 잘못 분류하는 이유**
가운데에 가로선이 있는 7은 아래쪽 구조가 2와 비슷해진다. 합성곱 신경망이 찾는 특징(가로선 + 대각선 조합)이 두 숫자에서 겹치기 때문이다. 게다가 MNIST 훈련 데이터에는 가로선이 있는 7이 드물어, 모델이 그 변형을 충분히 배우지 못한다.

**개선 방법**
1. **데이터 증강**: 가로선이 있는 7을 인위적으로 만들거나, 회전·이동 변환으로 7의 변형을 늘린다.
2. **모델 용량 확대**: 합성곱 블록을 늘려 더 세밀한 특징(7의 윗변 가로획과 2의 아랫변 곡선 차이)을 구분하게 한다.
3. **해당 클래스 가중치 조정**: 손실 함수의 `weight` 인자로 자주 틀리는 클래스에 더 큰 벌점을 준다.

가장 효과가 큰 것은 보통 **데이터 증강**이다. 문제의 원인이 모델 구조보다 데이터 분포의 편향에 있기 때문이다.

## 연습 5-5

예제의 합성곱 신경망 모델에는 padding=1, stride=1로 지정해 만든 두 개의 합성곱 계층이 포함되어 있다.

두 합성곱 계층을 만들 때 padding=0, stride=1로 인자의 값을 바꾸면, 모델에 포함된 합성곱 계층, 최대 풀링 계층, 평탄화 계층과 선형 계층의 입력과 출력 텐서의 형태는 어떻게 바뀔까?

이렇게 합성곱 계층을 바꾸면 모델 성능에 어떤 영향을 미치게 될까? 일반적인 경우와 MNIST 데이터셋을 학습하는 경우 각각 예상해 보자.

사용하는 합성곱 계층을 padding=0, stride=1인 합성곱 계층으로 바꾼 후 결과를 확인해 보자.

In [ ]:
# padding=1 과 padding=0 일 때 텐서 형태 변화를 비교한다.
def trace_shapes(padding):
    x = torch.randn(1, 1, 28, 28)
    print(f'  입력           {tuple(x.shape)}')
    for i in range(2):
        conv = nn.Conv2d(1 if i == 0 else 32, 32, kernel_size=3,
                         stride=1, padding=padding)
        x = conv(x); print(f'  합성곱 {i + 1}      {tuple(x.shape)}')
        x = nn.MaxPool2d(2)(x); print(f'  최대 풀링 {i + 1}   {tuple(x.shape)}')
    flat = x.flatten(1)
    print(f'  평탄화         {tuple(flat.shape)}  -> 선형 계층 입력 크기 {flat.shape[1]}')

for pad in (1, 0):
    print(f'padding={pad}')
    trace_shapes(pad)
    print()

`padding=0`이면 합성곱을 지날 때마다 가장자리가 깎여 특징 지도가 작아진다. 그 결과 평탄화 크기가 줄어 **선형 계층의 파라미터도 줄어든다**.

일반적으로는 가장자리 정보가 반복해서 손실되므로 깊은 신경망일수록 불리하다. 다만 MNIST는 숫자가 이미지 **가운데**에 있고 테두리가 거의 배경이라 성능 차이가 크지 않다. 그래서 이 예제에서는 padding=0으로 바꿔도 정확도가 비슷하게 유지된다.

## 연습 5-6

예제의 합성곱 신경망 모델에는 두 개의 합성곱 계층이 포함되어 있다. 합성곱 계층, 활성화 계층, 최대 풀링 계층을 하나 더 추가해 세 개의 합성곱 계층을 포함한 모델을 만들어 결과를 확인해 보자.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, fan_in, fan_out):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(fan_in, fan_out, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
    def forward(self, x): return self.block(x)

# 합성곱 블록 3개: 28 -> 14 -> 7 -> 3
torch.manual_seed(SEED)
model3 = nn.Sequential(
    ConvBlock(1, 32), ConvBlock(32, 64), ConvBlock(64, 128),
    nn.Flatten(), nn.Linear(128 * 3 * 3, 10),
)
print('합성곱 블록 3개 모델')
fit(model3, epochs=5)

블록을 하나 더 쌓으면 특징 지도가 7×7에서 3×3으로 더 줄고 채널은 128로 늘어난다. MNIST처럼 단순한 데이터에서는 블록 2개로도 충분해 성능 향상이 크지 않지만, 파라미터와 학습 시간은 늘어난다. **데이터 복잡도에 맞는 깊이**를 고르는 것이 중요하다.

## 연습 5-7

[코드 5-10]의 ConvBlock 클래스를 재사용하는 방식으로 합성곱 블록 두 개로 구성된 합성곱 신경망 클래스를 정의한 후 모델 객체를 생성해 보자. [코드 5-8]의 MNISTConvClassifier 클래스, [코드 5-9]의 MNISTConvClassifier_v2 클래스의 객체도 만든 다음, 세 모델 객체의 구조를 torchinfo.summary()와 print()로 출력해 보고 어떤 차이가 있는지 확인해 보자.

In [ ]:
import torchinfo

class ConvNetV3(nn.Module):
    """ConvBlock을 재사용해 만든 합성곱 신경망"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(ConvBlock(1, 32), ConvBlock(32, 64))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64 * 7 * 7, 10))
    def forward(self, x): return self.classifier(self.features(x))

model = ConvNetV3()
print('=== print() ===')
print(model)
print('\n=== torchinfo.summary() ===')
torchinfo.summary(model, input_size=(1, 1, 28, 28))

- **`print()`**: 모델을 구성하는 **모듈의 계층 구조**를 보여준다. 어떤 블록이 어떻게 중첩되어 있는지 파악하기 좋지만, 텐서 형태나 파라미터 수는 알 수 없다.
- **`torchinfo.summary()`**: 실제로 더미 입력을 흘려 보며 **계층별 출력 형태와 파라미터 수**까지 보여준다. 형태 오류를 찾을 때 유용하다.

블록으로 묶으면 `print()` 출력이 중첩되어 보이지만, 파라미터 수와 연산은 나열식으로 만든 모델과 완전히 같다.

## 연습 5-8

[도전 문제] 풀링 계층 없이 합성곱 계층만으로 (B, 1, 28, 28) 형태의 입력 텐서를 (B, 32, 7, 7) 형태로 줄인 후, 이를 분류기 계층으로 전달해 숫자를 분류하는 모델을 만들고, 결과를 확인해 보자. 단, 모델의 특징 추출기는 두 개의 합성곱 계층을 포함해야 한다.

In [ ]:
# 풀링 없이 stride=2 합성곱만으로 28 -> 14 -> 7 로 줄인다.
torch.manual_seed(SEED)
model_nopool = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1), nn.ReLU(),   # 28 -> 14
    nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1), nn.ReLU(),  # 14 -> 7
    nn.Flatten(), nn.Linear(32 * 7 * 7, 10),
)
x = torch.randn(1, 1, 28, 28)
print(f'특징 추출기 출력 형태: {tuple(model_nopool[:4](x).shape)}')
fit(model_nopool, epochs=5)

`stride=2`인 합성곱은 필터를 두 칸씩 건너뛰며 적용해 출력 크기를 절반으로 줄인다. 풀링과 달리 **줄이는 방법 자체를 학습**한다는 차이가 있다.

최대 풀링은 가장 강한 반응만 남기는 고정된 규칙이지만, stride 합성곱은 어떻게 요약할지 가중치로 배운다. 파라미터는 늘지만 표현력은 더 높다. 실제로 DCGAN(11장)의 판별자가 이 방식을 사용한다.